In [1]:
import pandas as pd
import numpy as np

print("Projet WC2026 ML lancé 🚀")


Projet WC2026 ML lancé 🚀


In [2]:
import pandas as pd
import numpy as np

# Charger le dataset
df = pd.read_csv("../data/results.csv")

# Afficher les 5 premières lignes
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [3]:
print(df.shape)

(49437, 9)


In [4]:
print(df.columns)

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='str')


In [5]:
df.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
40368,2016-12-26,Ivory Coast,Zimbabwe,0.0,0.0,Friendly,Abidjan,Ivory Coast,False
41506,2018-05-31,Mozambique,Seychelles,2.0,1.0,COSAFA Cup,Polokwane,South Africa,True
8198,1970-10-17,Switzerland,Italy,1.0,1.0,Friendly,Bern,Switzerland,False
14727,1985-04-12,Bahrain,Yemen DPR,3.0,3.0,FIFA World Cup qualification,Riffa,Bahrain,False
12053,1979-11-14,Malawi,Tanzania,4.0,0.0,CECAFA Cup,Nairobi,Kenya,True


In [6]:
df = df[[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament"
]]

In [7]:
df.sample(5)

,date,home_team,away_team,home_score,away_score,tournament
44090,2021-06-11,Malaysia,Vietnam,1.0,2.0,FIFA World Cup qualification
42425,2019-03-26,Romania,Faroe Islands,4.0,1.0,UEFA Euro qualification
21259,1996-08-16,Ecuador,Costa Rica,1.0,1.0,Friendly
29118,2005-03-30,Estonia,Russia,1.0,1.0,FIFA World Cup qualification
22078,1997-06-15,Mozambique,Tanzania,3.0,0.0,COSAFA Cup


In [8]:
def get_result(row):

    # Si l'équipe domicile marque plus
    if row["home_score"] > row["away_score"]:
        return 0

    # Si l'équipe extérieure marque plus
    elif row["home_score"] < row["away_score"]:
        return 2

    # Sinon match nul
    else:
        return 1

In [9]:
# Appliquer la fonction "get_result" à chaque ligne du DataFrame

In [10]:
df["result"] = df.apply(get_result, axis=1)

In [11]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,1
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,0


In [12]:
df["result"].value_counts()

result
0    24192
2    13948
1    11297
Name: count, dtype: int64

In [13]:
teams_stats = {}

In [14]:
for _, row in df.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    if home not in teams_stats:
        teams_stats[home] = {
            "matches": 0,
            "wins": 0
        }

    if away not in teams_stats:
        teams_stats[away] = {
            "matches": 0,
            "wins": 0
        }

    teams_stats[home]["matches"] += 1
    teams_stats[away]["matches"] += 1

    if row["result"] == 0:
        teams_stats[home]["wins"] += 1

    elif row["result"] == 2:
        teams_stats[away]["wins"] += 1

In [15]:
teams_stats["France"]

{'matches': 937, 'wins': 476}

In [16]:
for team in teams_stats:

    matches = teams_stats[team]["matches"]
    wins = teams_stats[team]["wins"]

    teams_stats[team]["win_rate"] = wins / matches

In [17]:
teams_stats["France"]

{'matches': 937, 'wins': 476, 'win_rate': 0.5080042689434365}

In [18]:
def get_team_win_rate(team):
    return teams_stats[team]["win_rate"]

In [19]:
df["home_win_rate"] = df["home_team"].apply(get_team_win_rate)

df["away_win_rate"] = df["away_team"].apply(get_team_win_rate)

In [20]:
df[[
    "home_team",
    "away_team",
    "home_win_rate",
    "away_win_rate",
    "result"
]].head()

,home_team,away_team,home_win_rate,away_win_rate,result
0,Scotland,England,0.470726,0.571429,1
1,England,Scotland,0.571429,0.470726,0
2,Scotland,England,0.470726,0.571429,0
3,England,Scotland,0.571429,0.470726,1
4,Scotland,England,0.470726,0.571429,0


In [21]:
X = df[[
    "home_win_rate",
    "away_win_rate"
]]

y = df["result"]

In [22]:
X.head()

,home_win_rate,away_win_rate
0,0.470726,0.571429
1,0.571429,0.470726
2,0.470726,0.571429
3,0.571429,0.470726
4,0.470726,0.571429


In [23]:
y.head()

0    1
1    0
2    0
3    1
4    0
Name: result, dtype: int64

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [25]:
print(X_train.shape)
print(X_test.shape)

(39549, 2)
(9888, 2)


In [26]:
from sklearn.linear_model import LogisticRegression

In [27]:
model = LogisticRegression(max_iter=1000)

In [28]:
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [29]:
print("Modèle entraîné avec succès")

Modèle entraîné avec succès


In [30]:
y_pred = model.predict(X_test)

In [31]:
y_pred[:10]

array([0, 0, 0, 2, 0, 0, 2, 0, 0, 0])

In [32]:
y_test.values[:10]

array([2, 2, 2, 0, 1, 0, 2, 2, 0, 0])

In [33]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.5614886731391586


In [34]:
accuracy_score(y_test, y_pred)

0.5614886731391586

In [35]:
from sklearn.metrics import confusion_matrix

In [36]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[4180    0  671]
 [1561    0  746]
 [1358    0 1372]]


In [37]:
from sklearn.metrics import classification_report

In [38]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.59      0.86      0.70      4851
           1       0.00      0.00      0.00      2307
           2       0.49      0.50      0.50      2730

    accuracy                           0.56      9888
   macro avg       0.36      0.45      0.40      9888
weighted avg       0.42      0.56      0.48      9888



/Users/omar/Prediction_World_Cup_2026_Winner-r-gression-logistique/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/omar/Prediction_World_Cup_2026_Winner-r-gression-logistique/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/omar/Prediction_World_Cup_2026_Winner-r-gression-logistique/venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predic